In [145]:
import torch
import matplotlib.pyplot as plt

In [146]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [147]:
device

'cuda'

In [148]:
!nvidia-smi

Sat Jul 25 16:48:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.62                 KMD Version: 610.62        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   42C    P8              3W /   44W |    6899MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [149]:
from datasets import load_dataset

In [150]:
raw_dataset = load_dataset("Helsinki-NLP/opus-100", "en-ja" , split = "train")

In [151]:
raw_dataset.shape

(1000000, 1)

In [152]:
raw_dataset[4]

{'translation': {'en': 'You okay?', 'ja': '無事か？'}}

In [153]:
shuffled_dataset = raw_dataset.shuffle(seed=42)

In [154]:
sample_dataset = shuffled_dataset.select(range(300000))
len(sample_dataset)

300000

In [155]:
sample_dataset[0]

{'translation': {'en': 'Kyle! No!', 'ja': 'ダメ！'}}

In [156]:
from torch.utils.data import DataLoader

splits = sample_dataset.train_test_split(test_size=0.2, seed=42)

train_dataset = splits['train']
val_dataset = splits['test']


In [157]:
len(train_dataset), len(val_dataset)

(240000, 60000)

In [158]:
import os

BATCH_SIZE = 16


train_dataLoader = DataLoader(train_dataset,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              num_workers=os.cpu_count())

val_dataLoader = DataLoader(val_dataset,
                            batch_size=BATCH_SIZE,
                            shuffle=False,
                            num_workers=os.cpu_count())

In [159]:
len(train_dataLoader), len(val_dataLoader)

(15000, 3750)

In [160]:
batch = next(iter(train_dataLoader))

In [161]:
batch.keys()

dict_keys(['translation'])

In [162]:
batch["translation"]

{'en': ['We believe Z.F.T. is a terrorist organization...',
  "You've both died in action.",
  'Guys, have a bowl and a Big Apple',
  'It was Chris Hargensen and all of her friends.',
  'Were you bad?',
  'Within 18 months of this picture, the modern environmental movement had begun.',
  "As a person who has lost a sibling, who has felt that pain, you won't survive if you're the one who kills her.",
  'Like they tried to kill lj.',
  "It's all part of the learning curve.",
  'Not so; for We have created them of what they know.',
  "- They're the key.",
  'It never occurred to him that I am the Blake association.',
  "Don't you guys have somewhere to be?",
  "Thank you. That's not my name.",
  'But the unjust changed the word that had been ordered for another one, so We sent down a punishment on them from the skies, the recompense of their disobedience.',
  'I think this belongs to Peppermint Patty.'],
 'ja': ['我々はZFTをテロリストの組織と見ている...',
  '野村 土屋 ２人ともまとめて殉職だ',
  '準備しろ ビッグアップルだ',
  'クリス・ハ

In [163]:
batch['translation']["en"][0],batch['translation']["ja"][0]

('We believe Z.F.T. is a terrorist organization...', '我々はZFTをテロリストの組織と見ている...')

In [129]:
from transformers import MarianTokenizer

tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-jap")

c:\Users\JAY\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [130]:
vocab_size = len(tokenizer.get_vocab())
n_embd = 512
block_size = 128
num_heads = 8
head_size = 64
num_encoder_blocks = 4
dropout = 0.1

In [131]:
from torch import nn
import torch.nn.functional as F

In [132]:
class Cross_Head(nn.Module):
    def __init__(self, head_size, n_embd):
        super().__init__()

        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output):

        B, T_dec, C = x.shape
        _, T_enc, _ = encoder_output.shape
        
       
        q = self.query(x)                 
        k = self.key(encoder_output)      
        v = self.value(encoder_output)    
        wei = q @ k.transpose(-2, -1) * (C ** -0.5)
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v                      
        return out

In [133]:
class Encoder_Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()

        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = F.softmax(wei , dim=-1)
        v = self.value(x)
        wei = self.dropout(wei)
        out = wei @ v
        return out


In [134]:
class MultiHeadAttention_encoder(nn.Module):
    def __init__(self, num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Encoder_Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads] , dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [135]:
class Decoder_Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()

        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei , dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

In [136]:
class MultiHeadAttention_decoder(nn.Module):
    def __init__(self, num_heads,head_size):
        super().__init__()
        self.heads = nn.ModuleList([Decoder_Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads] , dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

In [137]:
class MultiHeadAttention_cross(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, dropout=0.1):
        super().__init__()
        
        self.heads = nn.ModuleList([Cross_Head(head_size, n_embd) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output):
        
        out = torch.cat([h(x, encoder_output) for h in self.heads], dim=-1)
        out = self.proj(out)
        out  = self.dropout(out)
        return out

In [138]:
class FeedForward(nn.Module):
    def __init__(self,n_embd):
        super().__init__()

        self.net = nn.Sequential(
                    nn.Linear(n_embd ,4 * n_embd),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                    nn.Linear(4 * n_embd , n_embd),
                    nn.Dropout(dropout)
                    )

    def forward(self,x):
        return self.net(x)


In [139]:
class Encoder_Block(nn.Module):
    def __init__(self,num_heads,head_size,n_embd):
        super().__init__()
        self.sa = MultiHeadAttention_encoder(num_heads,head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self,x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [140]:
class Decoder_Block(nn.Module):
    def __init__(self,num_heads,head_size,n_embd):
        super().__init__()
        self.sa = MultiHeadAttention_decoder(num_heads,head_size)
        self.ca = MultiHeadAttention_cross(num_heads, head_size, n_embd)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x, encoder_output):
        x = x + self.sa(self.ln1(x))
        x = x + self.ca(self.ln2(x), encoder_output)
        x = x + self.ffwd(self.ln3(x))
        return x

In [141]:
class Translator(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.positonal_embedding_table = nn.Embedding(block_size, n_embd)

        self.encoder_blocks = nn.Sequential(*[Encoder_Block(num_heads,head_size,n_embd) for _ in range(num_encoder_blocks)])
        self.l1 = nn.LayerNorm(n_embd)

        self.decoder_blocks = nn.ModuleList([Decoder_Block(num_heads,head_size,n_embd) for _ in range(num_encoder_blocks)])
        self.l2 = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.dropout = nn.Dropout(dropout)


    def forward(self, src_ids, tgt_ids):

        B_src , T_src = src_ids.shape
        pos_src = torch.arange(T_src,device =src_ids.device)

        x_enc = self.token_embedding_table(src_ids) + self.positonal_embedding_table(pos_src)
        x_enc = self.dropout(x_enc)

        x_enc = self.encoder_blocks(x_enc) 

        encoder_output = self.l1(x_enc)

        B_tgt , T_tgt = tgt_ids.shape
        pos_tgt = torch.arange(T_tgt,device =tgt_ids.device)

        x_dec = self.token_embedding_table(tgt_ids) + self.positonal_embedding_table(pos_tgt)
        x_dec = self.dropout(x_dec)
        for block in self.decoder_blocks:
            x_dec = block(x_dec, encoder_output)
        x_dec = self.l2(x_dec)

        logits = self.lm_head(x_dec)
        return logits
        

In [142]:
model_0 = Translator().to(device)

device

'cuda'

In [143]:
dummy_src_ids = torch.randint(low=0, high=vocab_size, size=(BATCH_SIZE, block_size))
dummy_tgt_ids = torch.randint(low=0, high=vocab_size, size=(BATCH_SIZE, block_size))

In [144]:
loss = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model_0.parameters(), 
    lr=3e-4, 
    betas=(0.9, 0.98),  # Transformer-specific beta values
    eps=1e-9,           # Transformer-specific epsilon
    weight_decay=1e-4   # Standard L2 regularization
)

In [169]:
from torchmetrics.classification import MulticlassAccuracy
vocab_size = len(tokenizer.get_vocab())
pad_token_id = 0

token_accuracy = MulticlassAccuracy(
    num_classes=vocab_size, 
    ignore_index=pad_token_id,
    average='micro'
).to(device)



In [ ]:
from torchmetrics.text import BLEUScore

bleu_metric = BLEUScore().to(device)

In [ ]:
def train_step(model : torch.nn.Module,
               dataloader : torch.utils.data.DataLoader,
               loss_fn : torch.nn.Module,
               optimizer : torch.optim.Optimizer):
    
    model.train()

    train_loss = 0
    train_acc = 0

    for batch,(X,y) in enumerate(dataloader):

        X = X.to(device)
        y = y.to(device)

        y_pred = model(X)

        y_pred = y_pred.view(-1, y_pred.size(-1))
        y = y.view(-1)

        loss = loss_fn(y_pred,y)

        train_loss += loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)

        train_acc = token_accuracy(y_pred_class, y)

        print(f"---{batch}")

    train_loss /= len(dataloader)

    train_acc /= len(dataloader)

    return train_loss,train_acc

In [ ]:
def test_step(model : torch.nn.Module,
              dataloader : torch.utils.data.DataLoader,
              loss_fn : torch.nn.Module):
    model.eval()
    with torch.inference_mode():

        test_loss = 0
        test_acc = 0

        for X,y in dataloader:

            X = X.to(device)
            y = y.to(device)

            y_preds = model(X)

            y_pred = y_preds.view(-1, y_pred.size(-1))
            y_lo = y.view(-1)
            test_loss += loss_fn(y_pred,y_lo)

            test_pred_labels = y_preds.argmax(dim=1)
            
    
        test_loss /= len(dataloader)
        test_acc /= len(dataloader)

    return test_loss,test_acc

In [165]:
def train_model(model : torch.nn.Module,
                train_dataloader : torch.utils.data.DataLoader,
                test_dataloader : torch.utils.data.DataLoader,
                optimizer : torch.optim.Optimizer,
                loss_fn : torch.nn.Module,
                epochs : int = 5):
    
    results = {"train_loss" : [],
               "train_acc" : [],
               "test_loss" : [],
               "test_acc" : []
               }
    
    from tqdm.auto import tqdm

    for epoch in tqdm(range(epochs)):

        train_loss,train_acc = train_step(model = model,
                                dataloader= train_dataloader,
                                loss_fn= loss_fn,
                                optimizer= optimizer)
        
        test_loss,test_acc = test_step(model = model,
                                       dataloader=test_dataloader,
                                       loss_fn= loss_fn)
        
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}")
        

        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

    return results
